# 🏦 Bank Customer Churn Prediction
### IS-680 / IS-682 — Data Science & Data Mining

**Student:** Kumari Shivani Fnu  
**Dataset:** BankChurners for EDA 2024 (5,998 records, 21 features)  
**Goal:** Predict which bank customers are likely to churn (leave the bank)

---

## Project Phases
| Phase | Description |
|---|---|
| **Phase 1** | Exploratory Data Analysis (EDA) |
| **Phase 2** | Data Preprocessing + Model Training |
| **Phase 3** | Final Pre-Production Test on Unknown Records |

---

## Phase 1 — Exploratory Data Analysis (EDA)

Understanding the dataset before building any model.

In [ ]:
# Import all Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Read dataset
df = pd.read_csv("/content/drive/MyDrive/BankChurner/BankChurners for EDA 2024.csv")
print(f"Dataset shape: {df.shape}")
print(f"
Class distribution:")
print(df["Attrition Flag"].value_counts())
# Existing Customer: 5212, Attrited Customer: 786

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Data types
df.dtypes

In [ ]:
# Statistical summary
df.describe(include="all")

In [ ]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())
print("
Duplicate rows:", df.duplicated().sum())
# Result: 0 missing values, 0 duplicates

In [ ]:
# Garbage values in categorical columns
for col in df.select_dtypes(include="object").columns:
    print(f"
{col}:")
    print(df[col].value_counts())
    print("*" * 30)

In [ ]:
# Histograms - distribution of numeric columns
for col in df.select_dtypes(include="number").columns:
    sns.histplot(data=df, x=col)
    plt.title(f"Distribution of {col}")
    plt.show()

In [ ]:
# Boxplots - identify outliers
for col in df.select_dtypes(include="number").columns:
    sns.boxplot(data=df, x=col)
    plt.title(f"Boxplot of {col}")
    plt.show()

In [ ]:
# Scatter plots - relationship between features and Customer Age
numeric_cols = ["Dependent count", "Months on book", "Total Relationship Count",
                "Months Inactive 12 mon", "Contacts Count 12 mon", "Credit Limit",
                "Total Revolving Bal", "Avg Open To Buy", "Total Amt Chng Q4 Q1",
                "Total Trans Amt", "Total Trans Ct", "Total Ct Chng Q4 Q1", "Avg Utilization Ratio"]

for col in numeric_cols:
    sns.scatterplot(data=df, x=col, y="Customer Age", hue="Attrition Flag")
    plt.title(f"{col} vs Customer Age")
    plt.show()

---
## Phase 2 — Data Preprocessing & Model Training

In [ ]:
# Encode categorical target and gender
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["Attrition Flag"] = label_encoder.fit_transform(df["Attrition Flag"])
df["Gender"] = label_encoder.fit_transform(df["Gender"])
print("Encoding complete")
df[["Attrition Flag", "Gender"]].head()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(15, 15))
corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Outlier treatment using IQR Whisker method
def whisker(col):
    q1, q3 = np.percentile(col, [25, 75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in numeric_cols:
    lw, uw = whisker(df[col])
    df[col] = np.where(df[col] < lw, lw, df[col])
    df[col] = np.where(df[col] > uw, uw, df[col])

print("Outlier treatment complete")

In [ ]:
# One-hot encode categorical columns
df = pd.get_dummies(df, columns=["Education Level", "Marital Status",
                                   "Income Category", "Card Category"], dtype=int)
# Remove CLIENTNUM - just an ID, no predictive value
del df["CLIENTNUM"]
print(f"Final dataset shape: {df.shape}")

In [ ]:
# Spearman correlation with target
correlation = df.corr(method="spearman")["Attrition Flag"].sort_values()
print("Feature correlation with Attrition Flag:")
print(correlation)

In [ ]:
# Select top 7 features based on correlation analysis
selected_columns = ["Attrition Flag", "Total Relationship Count", "Total Trans Ct",
                    "Total Trans Amt", "Total Ct Chng Q4 Q1",
                    "Avg Utilization Ratio", "Months on book"]

df = df[selected_columns]
X = df.drop("Attrition Flag", axis=1)
y = df["Attrition Flag"]
print(f"Features: {X.columns.tolist()}")

In [ ]:
# Train / Test / Validation split (70% / 15% / 15%)
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Training:   {X_train.shape[0]:,} rows")
print(f"Testing:    {X_test.shape[0]:,} rows")
print(f"Validation: {X_val.shape[0]:,} rows")

In [ ]:
# Train baseline Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

y_pred_train = dt_model.predict(X_train)
y_pred_test  = dt_model.predict(X_test)
y_pred_val   = dt_model.predict(X_val)

print("=== Decision Tree Results ===")
print(f"Train - Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test  - Accuracy: {accuracy_score(y_test, y_pred_test):.4f} | Precision: {precision_score(y_test, y_pred_test):.4f} | Recall: {recall_score(y_test, y_pred_test):.4f}")
print(f"Val   - Accuracy: {accuracy_score(y_val, y_pred_val):.4f}")
# Test Accuracy: 0.9289, Precision: 0.6724, Recall: 0.75

In [ ]:
# 5-fold Cross-Validation
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(dt_model, X, y, cv=5, scoring="accuracy")
print("CV scores:", cv_scores)
print(f"Mean: {cv_scores.mean():.4f} | Std: {cv_scores.std():.4f}")
# Mean: 0.5921 - high variance, Decision Tree is unstable

---
## Hypothesis Testing

Testing whether Card Category and Marital Status are statistically significant predictors of churn.

In [ ]:
# Chi-square test
from scipy.stats import chi2_contingency

data = pd.read_csv("/content/drive/MyDrive/BankChurner/BankChurners for EDA 2024.csv")
contingency_table = pd.crosstab(data["Card Category"], data["Marital Status"])
chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"P-value:              {p_val:.6f}")
print(f"Degrees of freedom:   {dof}")
if p_val < 0.05:
    print("Result: Significant association (p < 0.05)")
else:
    print("Result: No significant association (p > 0.05)")

In [ ]:
# Cramers V - strength of association
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

v = cramers_v(pd.crosstab(data["Card Category"], data["Marital Status"]))
print(f"Cramers V: {v:.4f}")
# Value of 1.0 - strong association

---
## Phase 3 — Random Forest (Recommended Model)

Decision Tree had inconsistent cross-validation (mean 59.21%). Switching to Random Forest with GridSearchCV tuning.

In [ ]:
# Train Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== Random Forest ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf)*100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred_rf)*100:.2f}%")
print(f"Recall:    {recall_score(y_test, y_pred_rf)*100:.2f}%")
# Accuracy: 95.33%, Precision: 96.85%, Recall: 97.88%

In [ ]:
# GridSearchCV - hyperparameter tuning
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators":      [50, 100, 150, 200],
    "max_depth":         [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf":  [1, 2, 4, 6]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42),
                           param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
print("Best parameters:", best_params)
# Best: max_depth=None, min_samples_leaf=1, min_samples_split=10, n_estimators=200

In [ ]:
# Train optimized Random Forest with best parameters
best_rf = RandomForestClassifier(**best_params, random_state=42)
best_rf.fit(X_train, y_train)
y_pred_best = best_rf.predict(X_test)

print("=== Optimized Random Forest ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_best)*100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred_best)*100:.2f}%")
print(f"Recall:    {recall_score(y_test, y_pred_best)*100:.2f}%")

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred_best)
ConfusionMatrixDisplay(cm).plot(cmap="Blues")
plt.title("Confusion Matrix - Optimized Random Forest")
plt.show()

In [ ]:
# Save best model
from joblib import dump
dump(best_rf, "/content/drive/My Drive/BankChurner/best_rf_classifier.joblib")
print("Model saved to Google Drive")

---
## Phase 3 — Final Pre-Production Test

Applying model to 10 unknown records. Expecting ~20% attrition (2 out of 10).

In [ ]:
# Load and classify unknown Phase 3 records
phase3 = pd.read_csv("/content/drive/MyDrive/BankChurner/Phase_3_Records.csv")
phase3 = phase3.drop(columns=["CLIENTNUM"], errors="ignore")
phase3_encoded = pd.get_dummies(phase3)
phase3_encoded = phase3_encoded.reindex(columns=X.columns, fill_value=0)

predictions = best_rf.predict(phase3_encoded)
probabilities = best_rf.predict_proba(phase3_encoded)

results = pd.DataFrame({
    "Prediction": ["Attrited" if p == 0 else "Existing" for p in predictions],
    "Confidence": [f"{max(prob)*100:.1f}%" for prob in probabilities]
})
print(results)
print(f"
Attrited: {sum(predictions==0)}/10 ({sum(predictions==0)*10:.0f}%) — Expected: ~20%")

---
## Summary of Results

| Model | Accuracy | Precision | Recall |
|---|---|---|---|
| Decision Tree | 92.89% | 67.24% | 75.00% |
| **Random Forest (Final)** | **95.33%** | **96.85%** | **97.88%** |

### Why Random Forest was recommended:
- Much better precision and recall than Decision Tree
- Multiple trees reduce overfitting
- Consistent performance across all splits
- Correctly identifies ~20% attrition on unknown records

---
*IS-680/682 Project — Kumari Shivani Fnu*